# 🏗️ Capstone Project — Tax Compliance Advisor
### Days 11–12 · Team Build

---

## What you are building

A **3-agent system** that takes a client tax query and produces a cited compliance note.

```
Client input
      │
      ▼
┌─────────────┐
│  Validator  │  ← checks completeness, flags issues
└──────┬──────┘
       │
       ▼
┌─────────────┐
│ Researcher  │  ← searches tax law, returns citations
└──────┬──────┘
       │  (also runs the Day 7 classification pipeline)
       ▼
┌─────────────┐
│   Drafter   │  ← writes a 3-paragraph compliance note
└──────┬──────┘
       │
       ▼
  Compliance note + trace log
```

## Your three test scenarios

| # | Scenario | Expected behaviour |
|---|---|---|
| 1 | TechSolutions Mumbai→Bengaluru Rs 8.5L software | Clear case — Drafter produces a confident note |
| 2 | GlobalServ IT export USA, no LUT mentioned | Edge case — Validator flags LUT missing |
| 3 | 'Our vendor sent an invoice. Rs 3L. Please advise.' | Ambiguous — Validator flags incomplete input |

## Minimum requirements
- ✅ Validator flags at least 1 issue in Scenario 2 and Scenario 3
- ✅ Researcher returns at least 1 cited section for Scenario 1
- ✅ Drafter produces a readable 3-paragraph note for Scenario 1
- ✅ Trace CSV exported — at least 3 agent × 3 scenario rows

## Bonus targets
- ⭐ Conversation memory — follow-up question works in same session
- ⭐ Cost estimate — tokens and approximate cost per query
- ⭐ Confidence gating — if Validator flags too many issues, skip Drafter

---
### 🗓️  Day 11: Complete Steps 1–6 (Validator + Researcher + Supervisor scaffold)
### 🗓️  Day 12: Complete Steps 7–10 (Drafter + Memory + Demo prep)


## Step 1 — Setup

In [ ]:
%pip install openai chromadb python-dotenv --quiet

In [ ]:
import os, json, re, time, csv, hashlib
from datetime import datetime
from dotenv import load_dotenv
from openai import AzureOpenAI

load_dotenv()
client      = AzureOpenAI(
    azure_endpoint = os.getenv('AZURE_OPENAI_ENDPOINT'),
    api_key        = os.getenv('AZURE_OPENAI_API_KEY'),
    api_version    = '2024-08-01-preview',
)
CHAT_MODEL  = os.getenv('AZURE_OPENAI_CHAT_MODEL',  'gpt-4o')
EMBED_MODEL = os.getenv('AZURE_OPENAI_EMBED_MODEL', 'text-embedding-ada-002')

RATES = {
    'B2B Intra-State':    {'rate': 18.0, 'section': 'CGST Act §9'},
    'B2B Inter-State':    {'rate': 18.0, 'section': 'IGST Act §5'},
    'Export of Services': {'rate': 0.0,  'section': 'IGST Act §16(1)'},
    'Import RCM':         {'rate': 18.0, 'section': 'CGST Act §9(3)'},
    'Exempt':             {'rate': 0.0,  'section': 'CGST Schedule III'},
    'Works Contract':     {'rate': 18.0, 'section': 'Notification 11/2017'},
}

print('✅ Connected to Azure OpenAI')

## Step 2 — Knowledge Base

The Researcher agent searches this. If you have a ChromaDB store from Day 9, use it.
If not, the fallback knowledge base below will work for the demo.

In [ ]:
import chromadb

# ── Option A: Use your Day 9 ChromaDB store ────────────────────────────────
# DB_PATH = './chroma_db'   # point this at your Day 9 folder
# chroma  = chromadb.PersistentClient(path=DB_PATH)
# kb      = chroma.get_or_create_collection('tax_knowledge', metadata={'hnsw:space':'cosine'})
# print(f'Day 9 store loaded: {kb.count()} records')

# ── Option B: In-memory fallback (works even without Day 9 data) ───────────
chroma = chromadb.Client()   # in-memory, no persistence needed
kb     = chroma.get_or_create_collection('tax_knowledge', metadata={'hnsw:space':'cosine'})

# Seed with essential GST knowledge
TAX_LAW = [
    {'id':'igst_5',    'text':'IGST Act Section 5: Inter-state supply of services is subject to integrated tax at the rate notified by the government. For IT and software services, the rate is 18%.'},
    {'id':'igst_16',   'text':'IGST Act Section 16(1): Zero-rated supply means export of goods or services. A registered person making zero-rated supplies may claim refund of input tax credit subject to filing LUT (Letter of Undertaking) or bond.'},
    {'id':'cgst_9',    'text':'CGST Act Section 9: Central tax shall be levied on all intra-state supplies of goods or services at rates not exceeding 20%. For software services within the same state, CGST 9% and SGST 9% applies.'},
    {'id':'cgst_9_3',  'text':'CGST Act Section 9(3): Tax on reverse charge basis — services supplied by an advocate or a firm of advocates to a business entity are subject to reverse charge mechanism. The recipient pays the tax.'},
    {'id':'notif_11',  'text':'GST Notification 11/2017: Works contract services, when supplied to the government or local authority for construction of civil structures, are taxed at 12%. Other works contract services attract 18%.'},
    {'id':'schedule3', 'text':'CGST Schedule III: Activities which shall be treated neither as supply of goods nor supply of services. Healthcare services by clinical establishments and educational services by educational institutions are exempt from GST.'},
    {'id':'lut_rule',  'text':'GST Rule 96A: A registered person who intends to supply goods or services for export without payment of integrated tax shall furnish a Letter of Undertaking (LUT) in Form GST RFD-11. Without LUT, IGST at applicable rate must be paid.'},
    {'id':'gstr1',     'text':'GSTR-1: Monthly or quarterly return of outward supplies. B2B invoices above Rs 2.5L must be reported invoice-wise. Inter-state B2B supplies must be reported in Table 4. Exports must be reported in Table 6.'},
]

if kb.count() == 0:
    def embed(text):
        return client.embeddings.create(model=EMBED_MODEL, input=text.replace('\n',' ')).data[0].embedding

    kb.add(
        ids        = [r['id'] for r in TAX_LAW],
        documents  = [r['text'] for r in TAX_LAW],
        embeddings = [embed(r['text']) for r in TAX_LAW],
    )
    print(f'✅ Knowledge base seeded: {kb.count()} sections loaded')
else:
    print(f'✅ Knowledge base ready: {kb.count()} records')

## Step 3 — Supervisor State and Trace Log

Every agent writes its output into a shared `state` dict.
Every agent call also writes one row to the trace log.
**Do not modify this step — the scaffold is provided.**

In [ ]:
# ── Trace log — one row per agent per run ─────────────────────────────────
TRACE_LOG = []

def log_agent(scenario_id, agent_name, duration_ms, result_summary):
    TRACE_LOG.append({
        'timestamp':      datetime.now().isoformat(),
        'scenario_id':    scenario_id,
        'agent':          agent_name,
        'duration_ms':    round(duration_ms, 1),
        'result_summary': result_summary[:120],
    })


def make_state(scenario_id, query, amount=None):
    """Create a fresh state dict for one scenario."""
    return {
        'scenario_id':   scenario_id,
        'query':         query,
        'amount':        amount,
        # Validator output
        'issues':        [],
        'complete':      False,
        'missing_fields':[],
        # Researcher output
        'citations':     [],
        # Pipeline output (Day 7)
        'transaction_type': None,
        'confidence':    0.0,
        'rate':          None,
        'tax_amount':    None,
        'section':       None,
        'route':         None,
        # Drafter output
        'compliance_note': None,
        # Tracking
        'tokens_used':   0,
    }


# ── Helper: call the LLM ───────────────────────────────────────────────────
def llm(prompt, max_tokens=500, system=None):
    messages = []
    if system:
        messages.append({'role': 'system', 'content': system})
    messages.append({'role': 'user', 'content': prompt})
    response = client.chat.completions.create(
        model=CHAT_MODEL, messages=messages,
        temperature=0.0, max_tokens=max_tokens,
    )
    return response.choices[0].message.content, response.usage.total_tokens


print('✅ Scaffold ready — state dict, trace log, llm() helper defined')

## Step 4 — Agent 1: The Validator

**Job:** Check the client input for completeness before anything else runs.

**Input:** `state['query']` and `state['amount']`

**Output (write into state):**
- `state['issues']` — list of strings describing each problem found
- `state['complete']` — True if input is good enough to classify, False otherwise
- `state['missing_fields']` — list of field names that are absent

**Checks to implement:**
1. Is a transaction amount present (state['amount'] is not None)?
2. Are both a supplier and a recipient mentioned in the query?
3. Is the transaction type inferable (software, construction, export, etc.)?
4. If the query mentions 'export' or 'foreign client' — is LUT mentioned?

---
**Your task:** Complete the `validate()` function below.
Use one LLM call to extract the structured JSON. The prompt and JSON schema are provided.

In [ ]:
def validate(state):
    """
    Agent 1 — Validator.
    Checks completeness of client input. Updates state in-place. Returns state.
    """
    t0 = time.time()

    prompt = (
        f'Analyse this client tax query for completeness and return ONLY valid JSON.\n\n'
        f'Query: "{state["query"]}"\n'
        f'Amount provided: {state["amount"]}\n\n'
        'Check for:\n'
        '1. Is a transaction amount present?\n'
        '2. Are both a supplier and a recipient clearly identified?\n'
        '3. Is the nature of the transaction inferable (software, construction, export, etc.)?\n'
        '4. If the query mentions export or foreign client — is LUT (Letter of Undertaking) mentioned?\n\n'
        'Return JSON:\n'
        '{"complete": true/false, '
        '"issues": ["list of specific problems found"], '
        '"missing_fields": ["amount", "supplier", "recipient", "transaction_type", "LUT"]}\n\n'
        'Return {"complete": true, "issues": [], "missing_fields": []} if input is sufficient.'
    )

    # ── TODO: call llm(), parse the JSON, update state ─────────────────────
    # Hint: use the llm() helper defined in Step 3
    # Hint: strip markdown fences before json.loads()
    # Hint: state['complete'], state['issues'], state['missing_fields']

    # --- YOUR CODE HERE ---
    raw, tokens = llm(prompt, max_tokens=300)
    state['tokens_used'] += tokens

    try:
        cleaned = re.sub(r'^```(?:json)?\s*|\s*```$', '', raw.strip(), flags=re.MULTILINE)
        result  = json.loads(cleaned)
        state['complete']       = result.get('complete', False)
        state['issues']         = result.get('issues', [])
        state['missing_fields'] = result.get('missing_fields', [])
    except Exception:
        state['complete']       = False
        state['issues']         = ['Validator parse error — treating as incomplete']
        state['missing_fields'] = ['unknown']
    # --- END YOUR CODE ---

    duration = (time.time() - t0) * 1000
    summary  = f"complete={state['complete']}, issues={len(state['issues'])}"
    log_agent(state['scenario_id'], 'validator', duration, summary)
    return state


# Quick test
test_state = make_state('TEST', 'Our vendor sent an invoice. Rs 3L. Please advise.', amount=300000)
validate(test_state)
print(f'complete:  {test_state["complete"]}')
print(f'issues:    {test_state["issues"]}')
print(f'missing:   {test_state["missing_fields"]}')

## Step 5 — Agent 2: The Researcher

**Job:** Search the tax knowledge base for relevant sections.

**Input:** `state['query']`

**Output (write into state):**
- `state['citations']` — list of dicts: `[{'section_id': ..., 'text': ..., 'similarity': ...}]`

**How it works:**
1. Embed the query
2. Search the `kb` ChromaDB collection (top 3 results)
3. Store the results as citations

---
**Your task:** Complete the `research()` function below.

In [ ]:
def embed(text):
    return client.embeddings.create(model=EMBED_MODEL, input=text.replace('\n',' ')).data[0].embedding


def research(state, top_k=3):
    """
    Agent 2 — Researcher.
    Searches ChromaDB knowledge base for relevant tax law sections.
    Updates state['citations']. Returns state.
    """
    t0 = time.time()

    # ── TODO: embed the query, search kb, store results in state['citations'] ──
    # Hint: q_vec = embed(state['query'])
    # Hint: results = kb.query(query_embeddings=[q_vec], n_results=top_k)
    # Hint: state['citations'] is a list of dicts with keys: section_id, text, similarity

    # --- YOUR CODE HERE ---
    q_vec   = embed(state['query'])
    n       = min(top_k, kb.count())
    results = kb.query(query_embeddings=[q_vec], n_results=n)

    state['citations'] = [
        {
            'section_id': results['ids'][0][i],
            'text':       results['documents'][0][i],
            'similarity': round(1 - results['distances'][0][i], 3),
        }
        for i in range(len(results['ids'][0]))
    ]
    # --- END YOUR CODE ---

    duration = (time.time() - t0) * 1000
    top_id   = state['citations'][0]['section_id'] if state['citations'] else 'none'
    log_agent(state['scenario_id'], 'researcher', duration, f"{len(state['citations'])} citations, top={top_id}")
    return state


# Quick test
test_state2 = make_state('TEST2', 'TechSolutions Mumbai invoices Bengaluru client for software services Rs 8.5L', amount=850000)
research(test_state2)
print(f'Citations found: {len(test_state2["citations"])}')
for c in test_state2['citations']:
    print(f'  [{c["section_id"]}]  sim={c["similarity"]}  {c["text"][:60]}...')

## Step 6 — The Classification Pipeline (Day 7)

**This is provided — do not modify it.**
The pipeline runs inside the Supervisor after the Researcher.
It classifies the transaction, looks up the rate, and calculates tax.

In [ ]:
def run_pipeline(state):
    """
    Day 7 pipeline: classify → lookup_rate → calculate → route.
    Runs inside the Supervisor. Updates state in-place.
    """
    t0 = time.time()

    # Include top citation in prompt for better classification
    citation_context = ''
    if state['citations']:
        top = state['citations'][0]
        citation_context = f'\n\nRelevant tax section: {top["text"][:200]}'

    prompt = (
        f'Classify this invoice and return ONLY valid JSON.\n\n'
        f'Invoice: {state["query"]}{citation_context}\n\n'
        'JSON: {"transaction_type":"B2B Intra-State|B2B Inter-State|'
        'Export of Services|Import RCM|Exempt|Works Contract|Unknown",'
        '"confidence":0.0-1.0}'
    )
    raw, tokens  = llm(prompt, max_tokens=150)
    state['tokens_used'] += tokens

    try:
        cleaned = re.sub(r'^```(?:json)?\s*|\s*```$', '', raw.strip(), flags=re.MULTILINE)
        result  = json.loads(cleaned)
        state['transaction_type'] = result.get('transaction_type', 'Unknown')
        state['confidence']       = float(result.get('confidence', 0.3))
    except Exception:
        state['transaction_type'] = 'Unknown'
        state['confidence']       = 0.3

    # Lookup rate
    info = RATES.get(state['transaction_type'], {'rate': None, 'section': '—'})
    state['rate']    = info['rate']
    state['section'] = info['section']

    # Calculate tax
    if state['amount'] and state['rate'] is not None:
        state['tax_amount'] = round(state['amount'] * state['rate'] / 100, 2)

    # Route
    reasons = []
    if state['confidence'] < 0.78:              reasons.append('low confidence')
    if state['transaction_type'] == 'Unknown':   reasons.append('type unresolved')
    if state['issues']:                          reasons.append('validator flagged issues')
    state['route'] = 'human_review' if reasons else 'db_insert'

    duration = (time.time() - t0) * 1000
    log_agent(state['scenario_id'], 'pipeline', duration,
              f"{state['transaction_type']} conf={state['confidence']:.2f} route={state['route']}")
    return state


print('✅ Classification pipeline ready (Day 7)')

## Step 7 — Agent 3: The Drafter  ⬅ Day 12

**Job:** Write a 3-paragraph client-ready compliance note using all prior agent outputs.

**Input (from state):**
- Validator: `state['issues']`, `state['missing_fields']`
- Researcher: `state['citations']`
- Pipeline: `state['transaction_type']`, `state['rate']`, `state['tax_amount']`, `state['section']`

**Output (write into state):**
- `state['compliance_note']` — a 3-paragraph plain-English note

**Structure of the compliance note:**
1. **Classification** — what type of transaction this is and why, citing the relevant section
2. **Tax calculation** — rate applied, tax amount, total payable
3. **Filing requirement** — what the client needs to do next (GSTR-1, LUT, RCM payment, etc.)

**If Validator flagged issues:** paragraph 1 should flag what is missing before classifying.

---
**Your task:** Write the `draft()` function. Build the prompt carefully — include all outputs from prior agents.

In [ ]:
def draft(state):
    """
    Agent 3 — Drafter.
    Writes a 3-paragraph compliance note from all prior agent outputs.
    Updates state['compliance_note']. Returns state.
    """
    t0 = time.time()

    # ── TODO: Build the prompt and call llm() ──────────────────────────────
    # Your prompt should include:
    #   - The original query (state['query'])
    #   - Validator issues (state['issues']) and missing fields (state['missing_fields'])
    #   - Top 2 Researcher citations (state['citations'][:2])
    #   - Transaction type, rate, tax amount, section (state values)
    #   - Route decision (state['route'])
    # Your prompt should instruct:
    #   - Write exactly 3 paragraphs
    #   - Paragraph 1: classification (cite the section)
    #   - Paragraph 2: tax calculation
    #   - Paragraph 3: filing requirement and next steps
    #   - If issues exist: note them prominently in paragraph 1
    #   - Tone: professional, plain English, suitable for a client memo

    # --- YOUR CODE HERE ---
    citation_text = '\n'.join(
        f'- [{c["section_id"]}]: {c["text"][:150]}'
        for c in state['citations'][:2]
    ) if state['citations'] else 'No citations retrieved.'

    issues_text = ', '.join(state['issues']) if state['issues'] else 'None'

    tax_str = f'Rs {state["tax_amount"]:,.0f}' if state.get('tax_amount') is not None else 'not calculated (amount missing)'

    prompt = (
        f'You are a senior GST compliance advisor. Write a 3-paragraph client compliance note.\n\n'
        f'CLIENT QUERY: {state["query"]}\n\n'
        f'VALIDATOR FINDINGS:\n'
        f'  Input complete: {state["complete"]}\n'
        f'  Issues flagged: {issues_text}\n\n'
        f'RELEVANT TAX LAW SECTIONS:\n{citation_text}\n\n'
        f'CLASSIFICATION RESULT:\n'
        f'  Transaction type: {state["transaction_type"]}\n'
        f'  GST rate: {state["rate"]}%\n'
        f'  Tax amount: {tax_str}\n'
        f'  Governing section: {state["section"]}\n'
        f'  Routing decision: {state["route"]}\n\n'
        f'Write exactly 3 paragraphs:\n'
        f'Paragraph 1: State the classification, cite the governing section, and note any missing information the client must provide.\n'
        f'Paragraph 2: State the tax rate, calculate the liability, and explain who pays (supplier or RCM recipient).\n'
        f'Paragraph 3: State the filing requirement — which return (GSTR-1/GSTR-3B), whether LUT is needed, and any next steps.\n'
        f'Use professional plain English suitable for a client memo. Do not use bullet points.'
    )

    note, tokens = llm(prompt, max_tokens=500)
    state['tokens_used']     += tokens
    state['compliance_note']  = note
    # --- END YOUR CODE ---

    duration = (time.time() - t0) * 1000
    preview  = note[:80] if note else 'empty'
    log_agent(state['scenario_id'], 'drafter', duration, preview)
    return state


print('✅ Drafter defined — ready to test in Supervisor')

## Step 8 — The Supervisor

The Supervisor wires all agents together and handles the flow.
**This is provided — but read it carefully.**
Note: if the Validator flags the input as incomplete AND the route is human_review,
the Supervisor still runs the Drafter so the client gets a partial note explaining what's needed.

In [ ]:
def supervisor(scenario_id, query, amount=None):
    """
    Orchestrates all agents for one scenario.
    Returns the completed state dict.
    """
    print(f'\n{"="*60}')
    print(f'Scenario: {scenario_id}')
    print(f'Query:    {query[:80]}...' if len(query)>80 else f'Query:    {query}')
    print(f'{"="*60}')

    state = make_state(scenario_id, query, amount)

    # Agent 1: Validate
    print('\n[1/4] Validator...')
    state = validate(state)
    if state['issues']:
        print(f'  ⚠️  Issues: {state["issues"]}')
    else:
        print('  ✅ Input complete')

    # Agent 2: Research
    print('\n[2/4] Researcher...')
    state = research(state)
    for c in state['citations'][:2]:
        print(f'  📚 [{c["section_id"]}] sim={c["similarity"]} — {c["text"][:60]}...')

    # Day 7 Pipeline
    print('\n[3/4] Classification pipeline...')
    state = run_pipeline(state)
    print(f'  🏷️  Type: {state["transaction_type"]}  conf={state["confidence"]:.2f}  rate={state["rate"]}%  route={state["route"]}')

    # Agent 3: Draft
    print('\n[4/4] Drafter...')
    state = draft(state)
    print('  ✍️  Compliance note generated')

    print(f'\n📊 Total tokens: {state["tokens_used"]}  |  Trace entries: {len([r for r in TRACE_LOG if r["scenario_id"]==scenario_id])}')
    return state


print('✅ Supervisor ready')

## Step 9 — Test Scenario 1: Clear case

Run this first. It should produce a high-confidence classification and a complete compliance note.

In [ ]:
result1 = supervisor(
    scenario_id = 'S1',
    query       = 'TechSolutions Pvt Ltd (Mumbai) invoiced DataCore Ltd (Bengaluru) Rs 8,50,000 for software implementation services. Q3 FY2025.',
    amount      = 850000,
)

print('\n' + '─'*60)
print('COMPLIANCE NOTE:')
print('─'*60)
print(result1['compliance_note'])

## Step 10 — Test Scenario 2: Edge case (LUT missing)

The Validator should flag that LUT is not mentioned.
The Drafter should acknowledge this in paragraph 1.

In [ ]:
result2 = supervisor(
    scenario_id = 'S2',
    query       = 'GlobalServ India exports IT consulting services to Acme Corp USA. Invoice amount USD 15,000 (approx Rs 12,50,000). No mention of LUT.',
    amount      = 1250000,
)

print('\n' + '─'*60)
print('COMPLIANCE NOTE:')
print('─'*60)
print(result2['compliance_note'])

## Step 11 — Test Scenario 3: Ambiguous input

The Validator should flag multiple missing fields.
The Drafter should still produce something — a note explaining what is needed.

In [ ]:
result3 = supervisor(
    scenario_id = 'S3',
    query       = 'Our vendor sent us an invoice. Rs 3 lakhs. Please advise on GST.',
    amount      = 300000,
)

print('\n' + '─'*60)
print('COMPLIANCE NOTE:')
print('─'*60)
print(result3['compliance_note'])

## Step 12 — Export trace log and cost estimate

In [ ]:
# Export trace log to CSV
with open('capstone_traces.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['timestamp','scenario_id','agent','duration_ms','result_summary'])
    writer.writeheader()
    writer.writerows(TRACE_LOG)

print(f'✅ Exported {len(TRACE_LOG)} trace entries to capstone_traces.csv')
print()

# Print summary table
print(f'{"Scenario":<8} {"Agent":<12} {"Duration (ms)":>14} {"Result"}')
print('-' * 70)
for row in TRACE_LOG:
    print(f'{row["scenario_id"]:<8} {row["agent"]:<12} {row["duration_ms"]:>14.1f}  {row["result_summary"][:35]}')

print()

# Cost estimate
total_tokens = sum(r.get('tokens_used', 0) for r in [result1, result2, result3])
cost_usd     = (total_tokens / 1000) * 0.01   # rough average
print(f'Total tokens (3 scenarios): {total_tokens:,}')
print(f'Estimated cost:             ${cost_usd:.4f}')
print(f'Cost per query:             ${cost_usd/3:.4f}')

## Step 13 — Bonus: Add conversation memory ⭐ Day 12

After Scenario 1, a follow-up question arrives:
'Same vendor, new invoice Rs 3.2L for maintenance. Same treatment?'

Without memory: the Supervisor doesn't know what 'same vendor' refers to.
With memory: the session history tells it that TechSolutions Pvt Ltd, Mumbai → Bengaluru, was just discussed.

In [ ]:
# ── Simple session memory ───────────────────────────────────────────────────
SESSION_MEMORY = [
    {'role': 'system', 'content':
     'You are a senior GST compliance advisor. Use the conversation history '
     'to resolve references to previous invoices or clients.'}
]


def update_memory(user_msg, assistant_reply, max_turns=6):
    """Add exchange to session memory. Trim to last max_turns pairs."""
    SESSION_MEMORY.append({'role': 'user',      'content': user_msg})
    SESSION_MEMORY.append({'role': 'assistant', 'content': assistant_reply})
    non_sys = SESSION_MEMORY[1:]
    if len(non_sys) > max_turns * 2:
        SESSION_MEMORY[1:] = non_sys[-(max_turns * 2):]


def llm_with_memory(user_msg, max_tokens=500):
    """LLM call that includes full session history."""
    messages = SESSION_MEMORY + [{'role': 'user', 'content': user_msg}]
    resp     = client.chat.completions.create(
        model=CHAT_MODEL, messages=messages,
        temperature=0.0, max_tokens=max_tokens,
    )
    return resp.choices[0].message.content


# ── Test follow-up question ─────────────────────────────────────────────────
# Step 1: Tell the memory about Scenario 1
update_memory(
    user_msg        = 'TechSolutions Pvt Ltd (Mumbai) invoiced DataCore Ltd (Bengaluru) Rs 8,50,000 for software implementation.',
    assistant_reply = result1['compliance_note'][:300] if result1.get('compliance_note') else 'B2B Inter-State, 18% IGST, Rs 1,53,000 tax.',
)

# Step 2: Ask the follow-up
follow_up = 'Same vendor, new invoice Rs 3.2L for maintenance services. Same treatment?'
print('Follow-up question:', follow_up)
print()

response = llm_with_memory(follow_up)
update_memory(follow_up, response)

print('Response with memory:')
print(response)

---
## Demo Prep Checklist

Before your 5-minute demo, confirm:

- [ ] Scenario 1 runs end-to-end and produces a readable compliance note
- [ ] Scenario 2 shows the Validator flagging the LUT issue
- [ ] Scenario 3 shows the Validator flagging incomplete input
- [ ] `capstone_traces.csv` exists and has at least 9 rows (3 agents × 3 scenarios)
- [ ] You can point to the section reference in Scenario 1's compliance note
- [ ] (Bonus) Follow-up question in Step 13 resolves 'same vendor' correctly

## Demo script (5 minutes)

1. **Minute 1:** Introduce your team. Read Scenario 1 aloud.
2. **Minutes 2–3:** Run the supervisor cell for Scenario 1. Point at each agent's output as it prints.
3. **Minute 4:** Read the compliance note aloud. Point at the section reference.
4. **Minute 5:** One thing that worked well. One thing you'd improve with more time.
